
## Folder 04 — File 05: Silver Payments:


**Transformation rules:**

- Trim payment IDs, order IDs, transaction IDs, and payment methods.
- Standardize payment_status and payment_method to uppercase.
- Convert amount to DECIMAL(12,2).
- Convert payment_timestamp to TIMESTAMP.
- Validate payment status against SUCCESS, PENDING, FAILED, and REFUNDED.
- Quarantine missing identifiers, invalid amounts, timestamps, and statuses.
- Deduplicate using payment_id.

In [0]:
# ============================================================
# SILVER PAYMENTS - COMPLETE NOTEBOOK
# ============================================================


# 1. IMPORT LIBRARIES

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Libraries imported successfully.")


# 2. PROJECT CONFIGURATION

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"

BRONZE_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.bronze_payments"
)

SILVER_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_payments"
)

QUARANTINE_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.quarantine_payments"
)

ORDERS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_orders"
)

ALLOWED_PAYMENT_STATUSES = [
    "SUCCESS",
    "PENDING",
    "FAILED",
    "REFUNDED"
]

print("Project configuration completed.")


# 3. CREATE SILVER SCHEMA

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"`{CATALOG}`.`{SILVER_SCHEMA}`"
)

print("Silver schema is ready.")


# 4. READ BRONZE PAYMENTS

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise Exception(
        f"Bronze table not found: {BRONZE_TABLE}. "
        "Run Bronze Payments ingestion first."
    )

bronze_df = spark.table(BRONZE_TABLE)

bronze_count = bronze_df.count()

print("BRONZE PAYMENT RECORDS:", bronze_count)

display(bronze_df.limit(10))


# 5. VALIDATE SOURCE COLUMNS

required_columns = [
    "payment_id",
    "order_id",
    "payment_method",
    "payment_status",
    "transaction_id",
    "amount",
    "payment_timestamp"
]

missing_columns = [
    column
    for column in required_columns
    if column not in bronze_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required payment columns: {missing_columns}"
    )

print("Source column validation passed.")


# 6. CLEAN AND STANDARDIZE PAYMENT DATA

# Remove an old validation column if present.

if "_quarantine_reason" in bronze_df.columns:
    bronze_df = bronze_df.drop("_quarantine_reason")

cleaned_df = (
    bronze_df

    .withColumn(
        "payment_id",
        F.trim(F.col("payment_id").cast("string"))
    )

    .withColumn(
        "order_id",
        F.trim(F.col("order_id").cast("string"))
    )

    .withColumn(
        "payment_method",
        F.upper(F.trim(F.col("payment_method").cast("string")))
    )

    .withColumn(
        "payment_status",
        F.upper(F.trim(F.col("payment_status").cast("string")))
    )

    .withColumn(
        "transaction_id",
        F.trim(F.col("transaction_id").cast("string"))
    )

    .withColumn(
        "amount",
        F.expr("try_cast(amount AS DECIMAL(12,2))")
    )

    .withColumn(
        "payment_timestamp",
        F.expr("try_cast(payment_timestamp AS TIMESTAMP)")
    )
)

print("Payment standardization completed.")

display(cleaned_df.limit(10))


# 7. DEFINE PAYMENT DATA-QUALITY RULES

# Current business rules:
#
# 1. payment_id is mandatory.
# 2. order_id is mandatory.
# 3. payment_method is mandatory.
# 4. payment_status must be an allowed status.
# 5. amount must be present and non-negative.
# 6. payment_timestamp is mandatory.
#
# Zero amount is currently allowed.
# transaction_id is optional because we have not established
# that every payment status requires a transaction ID.


invalid_condition = (
    F.col("payment_id").isNull()
    | (F.col("payment_id") == "")

    | F.col("order_id").isNull()
    | (F.col("order_id") == "")

    | F.col("payment_method").isNull()
    | (F.col("payment_method") == "")

    | F.col("payment_status").isNull()
    | (~F.col("payment_status").isin(
        ALLOWED_PAYMENT_STATUSES
    ))

    | F.col("amount").isNull()
    | (F.col("amount") < 0)

    | F.col("payment_timestamp").isNull()
)


# 8. CREATE INVALID PAYMENT DATAFRAME

invalid_df = (
    cleaned_df

    .filter(invalid_condition)

    .withColumn(
        "_quarantine_reason",

        F.concat_ws(
            "; ",

            F.when(
                F.col("payment_id").isNull()
                | (F.col("payment_id") == ""),
                F.lit("Missing payment_id")
            ),

            F.when(
                F.col("order_id").isNull()
                | (F.col("order_id") == ""),
                F.lit("Missing order_id")
            ),

            F.when(
                F.col("payment_method").isNull()
                | (F.col("payment_method") == ""),
                F.lit("Missing payment_method")
            ),

            F.when(
                F.col("payment_status").isNull()
                | (~F.col("payment_status").isin(
                    ALLOWED_PAYMENT_STATUSES
                )),
                F.lit("Invalid payment_status")
            ),

            F.when(
                F.col("amount").isNull(),
                F.lit("Missing or invalid amount")
            ),

            F.when(
                F.col("amount") < 0,
                F.lit("Negative payment amount")
            ),

            F.when(
                F.col("payment_timestamp").isNull(),
                F.lit("Missing or invalid payment_timestamp")
            )
        )
    )
)


# 9. CREATE VALID PAYMENT DATAFRAME

valid_df = cleaned_df.filter(
    ~invalid_condition
)

valid_count = valid_df.count()
invalid_count = invalid_df.count()

print("VALID PAYMENT RECORDS:", valid_count)
print("INVALID PAYMENT RECORDS:", invalid_count)

display(
    invalid_df.select(
        "payment_id",
        "order_id",
        "payment_method",
        "payment_status",
        "transaction_id",
        "amount",
        "payment_timestamp",
        "_quarantine_reason"
    )
)


# 10. DETECT DUPLICATE PAYMENT RECORDS

# Assumption:
# One current record per payment_id.
#
# Multiple payments for the same order are allowed.
# We deduplicate by payment_id, NOT order_id.
#
# Prefer the latest ingestion metadata when available.
# If metadata is unavailable, use business columns
# as deterministic tie-breakers.

metadata_order_columns = []

if "_ingested_at" in valid_df.columns:
    metadata_order_columns.append(
        F.col("_ingested_at").desc_nulls_last()
    )

if "_ingestion_timestamp" in valid_df.columns:
    metadata_order_columns.append(
        F.col("_ingestion_timestamp").desc_nulls_last()
    )

if "_source_file" in valid_df.columns:
    metadata_order_columns.append(
        F.col("_source_file").asc_nulls_last()
    )

business_order_columns = [
    F.col("payment_timestamp").desc_nulls_last(),
    F.col("transaction_id").asc_nulls_last(),
    F.col("order_id").asc_nulls_last(),
    F.col("amount").desc_nulls_last(),
    F.col("payment_method").asc_nulls_last(),
    F.col("payment_status").asc_nulls_last()
]

deduplication_window = (
    Window
    .partitionBy("payment_id")
    .orderBy(
        *(metadata_order_columns + business_order_columns)
    )
)

ranked_df = valid_df.withColumn(
    "_row_number",
    F.row_number().over(deduplication_window)
)


# Keep first record per payment_id.

deduplicated_df = (
    ranked_df
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)


# Capture duplicate records for quarantine.

duplicate_df = (
    ranked_df
    .filter(F.col("_row_number") > 1)
    .drop("_row_number")
    .withColumn(
        "_quarantine_reason",
        F.lit("Duplicate payment_id")
    )
)

duplicate_count = duplicate_df.count()

deduplicated_count = deduplicated_df.count()

print("DUPLICATE PAYMENT RECORDS:", duplicate_count)
print("RECORDS AFTER DEDUPLICATION:", deduplicated_count)


# 11. COMBINE INVALID AND DUPLICATE RECORDS

quarantine_df = (
    invalid_df

    .unionByName(
        duplicate_df,
        allowMissingColumns=True
    )

    .withColumn(
        "quarantined_at",
        F.current_timestamp()
    )
)

quarantine_count = quarantine_df.count()

print("TOTAL QUARANTINE RECORDS:", quarantine_count)

display(
    quarantine_df.select(
        "payment_id",
        "order_id",
        "payment_status",
        "amount",
        "_quarantine_reason"
    )
)


# 12. WRITE SILVER PAYMENTS DELTA TABLE

(
    deduplicated_df

    .drop("_quarantine_reason")

    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print("Silver payments table written successfully.")


# 13. WRITE PAYMENT QUARANTINE DELTA TABLE

(
    quarantine_df

    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print("Payment quarantine table written successfully.")


# 14. READ BACK SILVER AND QUARANTINE TABLES

silver_df = spark.table(SILVER_TABLE)

quarantine_result_df = spark.table(
    QUARANTINE_TABLE
)

total_silver_records = silver_df.count()

total_quarantine_records = (
    quarantine_result_df.count()
)

print("SILVER PAYMENT RECORDS:", total_silver_records)
print("QUARANTINE PAYMENT RECORDS:", total_quarantine_records)


# 15. VALIDATE SILVER PAYMENT DATA

duplicate_payment_groups = (
    silver_df

    .groupBy("payment_id")

    .count()

    .filter(F.col("count") > 1)

    .count()
)

invalid_silver_records = (
    silver_df

    .filter(
        F.col("payment_id").isNull()
        | (F.col("payment_id") == "")

        | F.col("order_id").isNull()
        | (F.col("order_id") == "")

        | F.col("payment_method").isNull()
        | (F.col("payment_method") == "")

        | F.col("payment_status").isNull()
        | (~F.col("payment_status").isin(
            ALLOWED_PAYMENT_STATUSES
        ))

        | F.col("amount").isNull()
        | (F.col("amount") < 0)

        | F.col("payment_timestamp").isNull()
    )

    .count()
)

print("=" * 50)
print("SILVER PAYMENT VALIDATION")
print("=" * 50)

print("Silver records:", total_silver_records)
print("Duplicate payment groups:", duplicate_payment_groups)
print("Invalid Silver records:", invalid_silver_records)

if (
    duplicate_payment_groups > 0
    or invalid_silver_records > 0
):
    raise Exception(
        "Silver payment validation failed."
    )

print("Silver payment validation passed.")


# 16. BRONZE-TO-SILVER ID RECONCILIATION

bronze_payment_ids = (
    bronze_df

    .filter(
        F.col("payment_id").isNotNull()
        & (
            F.trim(
                F.col("payment_id").cast("string")
            ) != ""
        )
    )

    .select(
        F.trim(
            F.col("payment_id").cast("string")
        ).alias("payment_id")
    )

    .distinct()
)

silver_payment_ids = (
    silver_df

    .select("payment_id")

    .distinct()
)

quarantine_payment_ids = (
    quarantine_result_df

    .filter(F.col("payment_id").isNotNull())

    .select("payment_id")

    .distinct()
)

bronze_distinct_count = bronze_payment_ids.count()

silver_distinct_count = silver_payment_ids.count()


# IDs that exist in Bronze but not in Silver.

missing_payment_ids_df = (
    bronze_payment_ids

    .join(
        silver_payment_ids,
        on="payment_id",
        how="left_anti"
    )
)

missing_payment_ids_count = (
    missing_payment_ids_df.count()
)


# Missing Silver IDs that also have no quarantine explanation.

unexplained_missing_ids = (
    missing_payment_ids_df

    .join(
        quarantine_payment_ids,
        on="payment_id",
        how="left_anti"
    )

    .count()
)

print("=" * 50)
print("BRONZE-TO-SILVER PAYMENT RECONCILIATION")
print("=" * 50)

print("Bronze distinct payment IDs:", bronze_distinct_count)
print("Silver distinct payment IDs:", silver_distinct_count)
print("Missing payment IDs from Silver:", missing_payment_ids_count)
print("Unexplained missing payment IDs:", unexplained_missing_ids)

if unexplained_missing_ids > 0:

    print("Unexplained payment IDs:")

    display(
        missing_payment_ids_df.join(
            quarantine_payment_ids,
            on="payment_id",
            how="left_anti"
        )
    )

    raise Exception(
        "Payment ID reconciliation failed."
    )

print("Payment ID reconciliation passed.")


# 17. ROW COUNT RECONCILIATION

reconciled_count = (
    total_silver_records
    + total_quarantine_records
)

print("=" * 50)
print("PAYMENT ROW COUNT RECONCILIATION")
print("=" * 50)

print("Bronze records:", bronze_count)
print("Silver records:", total_silver_records)
print("Quarantine records:", total_quarantine_records)
print("Silver + Quarantine:", reconciled_count)

if reconciled_count != bronze_count:
    raise Exception(
        "Payment row count reconciliation failed."
    )

print("PAYMENT ROW COUNT RECONCILIATION PASSED.")


# 18. REFERENTIAL INTEGRITY - ORDER VALIDATION

if not spark.catalog.tableExists(ORDERS_TABLE):
    raise Exception(
        f"Silver Orders table not found: {ORDERS_TABLE}"
    )

orders_df = spark.table(ORDERS_TABLE)

payment_order_ids = (
    silver_df

    .select("order_id")

    .distinct()
)

valid_order_ids = (
    orders_df

    .select("order_id")

    .distinct()
)

payments_without_orders_df = (
    payment_order_ids

    .join(
        valid_order_ids,
        on="order_id",
        how="left_anti"
    )
)

payments_without_orders_count = (
    payments_without_orders_df.count()
)

print(
    "PAYMENTS WITHOUT MATCHING SILVER ORDERS:",
    payments_without_orders_count
)

display(
    payments_without_orders_df.limit(30)
)


# 19. PAYMENT STATUS ANALYSIS

payment_status_summary = (
    silver_df

    .groupBy("payment_status")

    .agg(
        F.count("*").alias("total_transactions"),

        F.sum("amount").alias("total_amount"),

        F.round(
            F.avg("amount"),
            2
        ).alias("average_payment_amount")
    )

    .orderBy("payment_status")
)

display(payment_status_summary)


# 20. PAYMENT METHOD ANALYSIS

payment_method_summary = (
    silver_df

    .groupBy("payment_method")

    .agg(
        F.count("*").alias("total_transactions"),

        F.sum("amount").alias("total_amount"),

        F.round(
            F.avg("amount"),
            2
        ).alias("average_payment_amount")
    )

    .orderBy(F.desc("total_amount"))
)

display(payment_method_summary)


# 21. ORDER-LEVEL PAYMENT SUMMARY

# Multiple payments for one order are valid.
# This is an analysis table, not a deduplication step.

order_payment_summary = (
    silver_df

    .groupBy("order_id")

    .agg(
        F.count("*").alias("payment_count"),

        F.sum(
            F.when(
                F.col("payment_status") == "SUCCESS",
                F.col("amount")
            ).otherwise(F.lit(0))
        ).alias("successful_payment_amount"),

        F.sum(
            F.when(
                F.col("payment_status") == "REFUNDED",
                F.col("amount")
            ).otherwise(F.lit(0))
        ).alias("refunded_amount"),

        F.sum(
            F.when(
                F.col("payment_status") == "PENDING",
                F.col("amount")
            ).otherwise(F.lit(0))
        ).alias("pending_amount")
    )

    .orderBy("order_id")
)

display(order_payment_summary.limit(20))


# 22. PAYMENT QUARANTINE REASON ANALYSIS

quarantine_reason_summary = (
    quarantine_result_df

    .groupBy("_quarantine_reason")

    .agg(
        F.count("*").alias("record_count")
    )

    .orderBy(F.desc("record_count"))
)

display(quarantine_reason_summary)


# 23. DISPLAY FINAL SILVER PAYMENT DATA

display(
    silver_df

    .orderBy("payment_timestamp")

    .limit(20)
)


# 24. FINAL PAYMENT SUMMARY

final_bronze_count = spark.table(
    BRONZE_TABLE
).count()

final_silver_count = spark.table(
    SILVER_TABLE
).count()

final_quarantine_count = spark.table(
    QUARANTINE_TABLE
).count()

print("=" * 50)
print("SILVER PAYMENTS - FINAL SUMMARY")
print("=" * 50)

print("Bronze Records:", final_bronze_count)
print("Silver Records:", final_silver_count)
print("Invalid Records:", invalid_count)
print("Duplicate Records:", duplicate_count)
print("Total Quarantine Records:", final_quarantine_count)

print(
    "Payment IDs Without Silver Records:",
    missing_payment_ids_count
)

print(
    "Unexplained Missing Payment IDs:",
    unexplained_missing_ids
)

print(
    "Payments Without Matching Orders:",
    payments_without_orders_count
)

print("=" * 50)

if (
    final_silver_count + final_quarantine_count
    == final_bronze_count
):
    print("ROW COUNT RECONCILIATION PASSED.")
else:
    raise Exception(
        "Final payment row count reconciliation failed."
    )

print("Silver Payments notebook execution completed.")


In [0]:
orders_df = spark.table(
    "workspace.ecommerce_silver.silver_orders"
)

payments_df = spark.table(
    "workspace.ecommerce_silver.silver_payments"
)

items_df = spark.table(
    "workspace.ecommerce_silver.silver_order_items"
)

shipments_df = spark.table(
    "workspace.ecommerce_silver.silver_shipments"
)

payment_order_ids = payments_df.select("order_id").distinct()
item_order_ids = items_df.select("order_id").distinct()
shipment_order_ids = shipments_df.select("order_id").distinct()

missing_order_ids = (
    payment_order_ids
    .join(
        orders_df.select("order_id").distinct(),
        "order_id",
        "left_anti"
    )
)

display(
    missing_order_ids
    .join(item_order_ids, "order_id", "inner")
    .join(shipment_order_ids, "order_id", "inner")
)


### Verify the IDs against Bronze Orders:

In [0]:
from pyspark.sql import functions as F

bronze_orders = spark.table(
    "workspace.ecommerce_bronze.bronze_orders"
)

print("Bronze order count:", bronze_orders.count())

print("Matching order IDs:", 
    bronze_orders
    .filter(F.col("order_id").isin(missing_order_ids))
    .count()
)

print("Distinct Bronze order IDs:",
    bronze_orders.select("order_id").distinct().count()
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

bronze_orders = spark.table(
    "workspace.ecommerce_bronze.bronze_orders"
)

# Explicit schema
schema = StructType([
    StructField("order_id", StringType(), True)
])

# IDs present in Bronze
matched_ids = (
    bronze_orders
    .filter(F.col("order_id").isin(missing_order_ids))
    .select("order_id")
    .distinct()
)

# Create DataFrame with explicit schema
input_ids_df = spark.createDataFrame(
    [(str(x),) for x in missing_order_ids],
    schema
)

# IDs absent from Bronze
unmatched_ids = (
    input_ids_df
    .join(
        matched_ids,
        on="order_id",
        how="left_anti"
    )
)

print("IDs present in Bronze:")
display(matched_ids.orderBy("order_id"))

print("IDs absent from Bronze:")
display(unmatched_ids.orderBy("order_id"))

print("Present in Bronze:", matched_ids.count())
print("Absent from Bronze:", unmatched_ids.count())

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_silver.silver_payments;